In [16]:
# 1. Inicialización del Clúster y Estructuración del Pipeline ETL para Quebec

In [17]:
import os
import warnings
import polars as pl
import dask.dataframe as dd
from dask.distributed import Client

warnings.filterwarnings('ignore')

# OJO OJO Eliges qué archivo quieres procesar
#RUTA_FIRMS_GLOBAL = "../data/historial_quebec.parquet"
RUTA_FIRMS_GLOBAL = "../data/nrt_quebec.parquet"


# Inicialización del Clúster
try:
    cliente_activo = Client.current()
    cliente_activo.close()
except ValueError:
    pass

try:
    client = Client(n_workers=6, threads_per_worker=4, memory_limit='2.5GB')
    print("--- CLÚSTER DASK LOCAL EN KABRÉ INICIALIZADO ---")
except Exception as e:
    print(f"Nota sobre Dask: {e}")

print(f"Versión de Polars lista en el Kernel: {pl.__version__}")

# Parámetros geográficos
LAT_MIN, LAT_MAX = 45.0, 63.0
LON_MIN, LON_MAX = -80.0, -57.0

# 2. Ejecución del Pipeline
if os.path.exists(RUTA_FIRMS_GLOBAL):
    print(f"\n[INFO] Archivo detectado: {RUTA_FIRMS_GLOBAL}. Creando consulta Lazy...")
    
    consulta_lazy = (
        pl.scan_parquet(RUTA_FIRMS_GLOBAL)
        .filter(
            (pl.col("latitude") >= LAT_MIN) & (pl.col("latitude") <= LAT_MAX) &
            (pl.col("longitude") >= LON_MIN) & (pl.col("longitude") <= LON_MAX)
        )
        .filter(
            (pl.col("confidence").cast(pl.Utf8) != "low")
        )
        .with_columns([
            pl.col("acq_date").str.to_date(),
        ])
    )
    
    print("[INFO] Ejecutando procesamiento paralelo (Streaming activado)...")
    df_quebec = consulta_lazy.collect(streaming=True)
    
    print(f"\n¡Preprocesamiento completado exitosamente!")
    print(f"Registros reducidos a la zona de Quebec: {df_quebec.shape[0]}")
    
    RUTA_SALIDA = "firms_quebec_limpio.parquet"
    df_quebec.write_parquet(RUTA_SALIDA)
    print(f"[ÉXITO] Archivo listo en: {RUTA_SALIDA}")
    
else:
    print(f"\n[ERROR] El archivo no se encuentra en la ruta: {RUTA_FIRMS_GLOBAL}")
    print("Verifica que el nombre del archivo y la carpeta sean correctos.")

--- CLÚSTER DASK LOCAL EN KABRÉ INICIALIZADO ---
Versión de Polars lista en el Kernel: 1.36.1

[INFO] Archivo detectado: ../data/nrt_quebec.parquet. Creando consulta Lazy...
[INFO] Ejecutando procesamiento paralelo (Streaming activado)...

¡Preprocesamiento completado exitosamente!
Registros reducidos a la zona de Quebec: 22460
[ÉXITO] Archivo listo en: firms_quebec_limpio.parquet


In [18]:
# 2. Generación del Dataset de Simulación Masiva para Quebec

In [19]:
import polars as pl
import numpy as np

n_filas = 50_000_000

df_quebec_falso = pl.DataFrame({
    "latitude": np.random.uniform(45.0, 62.5, n_filas).astype(np.float32),
    "longitude": np.random.uniform(-79.0, -57.0, n_filas).astype(np.float32),
    "frp": np.random.uniform(5.0, 500.0, n_filas).astype(np.float32),
    "confidence": np.random.randint(0, 101, size=n_filas, dtype=np.int32),
    "satellite": np.random.choice(["Aqua", "Terra", "VIIRS"], size=n_filas)
})

fechas_base = np.datetime64('2025-01-01') + np.random.randint(0, 365, n_filas).astype('timedelta64[D]')
df_quebec_falso = df_quebec_falso.with_columns(pl.Series("acq_date", fechas_base))

ruta_falsa_global = "firms_global_SIMULADO.parquet"
df_quebec_falso.write_parquet(ruta_falsa_global)

In [20]:
# 3. Filtrado Geográfico y Selección de Columnas Críticas

In [21]:
import polars as pl

df_real = pl.read_parquet("firms_global_SIMULADO.parquet")

pipeline_resultado = (
    df_real
    .filter(
        (pl.col("latitude").is_between(46.0, 52.0)) & 
        (pl.col("longitude").is_between(-75.0, -68.0))
    )
    .filter(pl.col("confidence") >= 80) 
    .select(["acq_date", "latitude", "longitude", "frp", "satellite"])
)

pipeline_resultado.write_parquet("firms_quebec_filtrado.parquet")
print(pipeline_resultado.shape)

(1135470, 5)
